# ED tokenizers

Declare the alphabet, import the tokenizers, and train them. Bigram vocabularies come from **IAM training data only**; ED supplies its declared alphabet. Outputs are shared across WD/WI folds. Run from the TVA root with the WSL tva kernel.


In [ ]:
from pathlib import Path
from tva.handwriting_tokenizers import CharacterTokenizer, IAMBigramTokenizer
from tva.handwriting_tokenizers import GreedyHandwritingBigramTokenizer

# Run from the TVA repository root with the WSL tva kernel.
iam_labels = Path('/home/artellisys/DTLR/data/IAM_new/labels.pkl')
iam_selection = Path('/home/artellisys/dtlr-output/iam-train-full/selection.json')
iam_handwriting = Path('artifacts/tokenizers/source/iam-english-handwriting-bigram-v1.json')
output = Path('artifacts/tokenizers')

categories = [
    '', ' ', '!', '"', '%', "'", '(', ')', '+', ',', '-', '.', '/',
    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '=', '?',
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M',
    'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z',
    'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm',
    'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z',
]


## Character tokenizer

The character baseline needs only the declared alphabet. Its config already contains these categories, so no separate character file or learned vocabulary is needed.


In [ ]:
tokenizer_char = CharacterTokenizer()
tokenizer_char.load_categories(categories)
print('Character classes:', tokenizer_char.size)


## Linguistic bigram — original TVA on IAM

The wrapper prepares IAM input and calls the unchanged TVA BigramTokenizer.train(): unique complete labels, unrestricted adjacent pairs, frequency selection, greedy encoding. Source checks and a fixed hash seed are handled inside the package.


In [ ]:
tokenizer_bigram = IAMBigramTokenizer()
tokenizer_bigram.train(
    iam_labels, categories, 224,
    selection_path=iam_selection,
    output_path=output / 'ed_iam_tva_original_bigram_greedy_v1.json',
)
print('Linguistic classes:', tokenizer_bigram.size)


## Handwriting bigram — frozen IAM evidence

Here .train() prepares the ED vocabulary from DTLR’s already-frozen IAM handwriting evidence. It preserves all 145 bigrams and adapts the singleton alphabet/IDs. It does not retrain the handwriting evidence model or read ED labels.


In [ ]:
tokenizer_handwriting = GreedyHandwritingBigramTokenizer()
tokenizer_handwriting.train(
    iam_handwriting, categories,
    output_path=output / 'ed_iam_handwriting_bigram_greedy_v1.json',
)
print('Handwriting classes:', tokenizer_handwriting.size)


## Encode and decode


In [ ]:
text = 'In short, there will be on TV!'
for tokenizer in (tokenizer_char, tokenizer_bigram, tokenizer_handwriting):
    ids = tokenizer.encode(text)
    print(type(tokenizer).__name__, ids, tokenizer.decode(ids))


Both bigram files have **224 classes**. Existing files are accepted only when identical, preserving completed experiments.

Detailed checks live outside this notebook:
~~~bash
python -m tva.handwriting.audit --compare --dataset-directory /mnt/c/Users/Ali/Downloads/fau-english-dataset
~~~

Recognition-model training remains separate and manual, using configs/thesis/ed/.
